# 03 — LDA experiments

E2: PCA(m) → LDA + 3 classifiers, d = 1 … c − 1. Default m = N − c; optional sweep over m.

In [ ]:
import sys, pathlib
sys.path.append(str(pathlib.Path.cwd().parent))
%load_ext autoreload
%autoreload 2

In [ ]:
import warnings

import numpy as np

from src.classifiers import min_euclid_predict, min_mahal_predict
from src.config import OFFSETS, RESULTS_DIR
from src.data import load_coil20, make_split
from src.evaluate import CLASSIFIERS, run_lda_experiment
from src.reduction import fit_lda, fit_pca, pca_project

ARRAYS = RESULTS_DIR / "arrays"
ARRAYS.mkdir(parents=True, exist_ok=True)
names = list(CLASSIFIERS)

In [ ]:
X, y, pose = load_coil20()

## LDA sanity checks

On every split with m = N − c: $W^T S_w W = I$, $S_b W = S_w W \Lambda$, eigenvalues sorted, exactly c − 1 non-zero generalised eigenvalues. Tolerances are relative because $S_w$ is badly conditioned at m = N − c.

In [ ]:
for off in OFFSETS:
    tr, te = make_split(pose, off)
    N, c = len(tr), len(np.unique(y[tr]))
    m, V, _ = fit_pca(X[tr])
    Ztr = pca_project(X[tr], m, V, N - c)
    W, lam = fit_lda(Ztr, y[tr])
    M = np.stack([Ztr[y[tr] == k].mean(axis=0) for k in range(c)])
    R = Ztr - M[y[tr]]
    Sw = R.T @ R
    D = M - Ztr.mean(axis=0)
    Sb = (D.T * np.bincount(y[tr])) @ D
    assert W.shape == (N - c, c - 1) and np.all(np.diff(lam) <= 0)
    assert np.abs(W.T @ Sw @ W - np.eye(c - 1)).max() < 1e-6
    assert np.linalg.norm(Sb @ W - Sw @ W * lam) / np.linalg.norm(Sb @ W) < 1e-6
    assert np.linalg.matrix_rank(Sb) == c - 1
    print(f"split {off}: cond(S_w) = {np.linalg.cond(Sw):.1e}, lambda = {lam[0]:.3g} … {lam[-1]:.3g}")
print("LDA checks passed")

Cross-checks on split 0 (sklearn for checking only):

- our LDA gives the same min-Euclid accuracy as `sklearn` LDA (`solver="eigen"`);
- in LDA space min-Mahal = min-Euclid, because $W^T S_w W = I$ makes $\Sigma_w \propto I$;
- with all c − 1 directions, min-Euclid in LDA space = min-Mahal in the m-dim PCA space.

In [ ]:
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis   # cross-check only

tr, te = make_split(pose, 0)
N, c = len(tr), len(np.unique(y[tr]))
m, V, _ = fit_pca(X[tr])
for mp in (50, N - c):
    Ztr, Zte = pca_project(X[tr], m, V, mp), pca_project(X[te], m, V, mp)
    W, _ = fit_lda(Ztr, y[tr])
    Ltr, Lte = Ztr @ W, Zte @ W
    sk = LinearDiscriminantAnalysis(solver="eigen").fit(Ztr, y[tr])
    eu = min_euclid_predict(Ltr, y[tr], Lte)
    assert (eu == min_euclid_predict(sk.transform(Ztr), y[tr], sk.transform(Zte))).all()
    assert (eu == min_mahal_predict(Ltr, y[tr], Lte)).all()
    assert (eu == min_mahal_predict(Ztr, y[tr], Zte)).all()
    print(f"m = {mp}: cross-checks passed, accuracy {np.mean(eu == y[te]):.4f}")

## E2: PCA(N − c) → LDA, d = 1 … c − 1

In [ ]:
d_list = np.arange(1, 20)
acc = run_lda_experiment(X, y, pose, d_list)
np.savez(ARRAYS / "lda_acc.npz", acc=acc, d_list=d_list, classifiers=names)
mean, std = acc.mean(axis=0), acc.std(axis=0)
for d in (1, 2, 5, 10, 15, 19):
    print(f"d={d:2d}  " + "  ".join(f"{n} {mean[i, d - 1]:.4f}±{std[i, d - 1]:.4f}" for i, n in enumerate(names)))
print("per split, 1-NN at d = 19:", np.round(acc[:, 0, -1], 4))

## E2 (optional): sweep the intermediate PCA dimension m

m ≥ c − 1 = 19 and m ≤ N − c = 220. This shows test curves only; choosing m is done by training-only validation (notebook 04).

In [ ]:
m_list = np.array([20, 30, 50, 75, 100, 150, 200, 220])
acc_m = np.stack([run_lda_experiment(X, y, pose, d_list, m_pca=mp) for mp in m_list])
np.savez(ARRAYS / "lda_m_sweep.npz", acc=acc_m, m_list=m_list, d_list=d_list, classifiers=names)
print(acc_m.shape)   # [m_index, offset, classifier, d_index]
for mp, a in zip(m_list, acc_m):
    print(f"m={mp:3d}  d=19: " + "  ".join(f"{n} {a[:, i, -1].mean():.4f}±{a[:, i, -1].std():.4f}" for i, n in enumerate(names)))

### Quick look (not a report figure)

In [ ]:
import matplotlib.pyplot as plt

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
for i, n in enumerate(names):
    ax1.plot(d_list, 1 - mean[i], marker="o", ms=3, label=n)
ax1.set(xlabel="d", ylabel="error rate", title="E2: PCA(N − c) → LDA", xticks=d_list[::2])
ax1.legend()
for i, n in enumerate(names):
    ax2.plot(m_list, 1 - acc_m[:, :, i, -1].mean(axis=1), marker="o", label=n)
ax2.set(xlabel="intermediate PCA dim m", ylabel="error rate at d = 19", title="E2: effect of m")
ax2.legend()
plt.show()